In [7]:
import os
import time
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter

import torchvision.transforms as transforms
from albumentations import Compose, Resize, Normalize, HorizontalFlip, RandomBrightnessContrast
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from efficientnet_pytorch import EfficientNet

# Check PyTorch and CUDA
print("="*70)
print("SYSTEM INFORMATION")
print("="*70)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*70)


SYSTEM INFORMATION
PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
GPU Memory: 8.59 GB


In [8]:
# ========================================================================
# EXTRACT DATASET
# ========================================================================

# Update this path to your zip file location
zip_path = "path/to/your/nih_chest_xray.zip"  # CHANGE THIS
extract_path = "./chest_xray_data/"

# Create directory
os.makedirs(extract_path, exist_ok=True)

# Extract (skip if already done)
if not os.path.exists(os.path.join(extract_path, "Data_Entry_2017.csv")):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset extracted successfully!")
else:
    print("✅ Dataset already extracted!")

# Check dataset size
def get_folder_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024**3)  # GB

dataset_size = get_folder_size(extract_path)
print(f"\n📊 Dataset size: {dataset_size:.2f} GB")

if dataset_size > 30:
    print("⚠️ LARGE DATASET DETECTED!")
    print("   Optimizations will be applied for faster training")


✅ Dataset already extracted!

📊 Dataset size: 41.98 GB
⚠️ LARGE DATASET DETECTED!
   Optimizations will be applied for faster training


In [9]:
# ========================================================================
# LOAD AND EXPLORE DATASET
# ========================================================================

# Load data entry file
data_entry_path = os.path.join(extract_path, "Data_Entry_2017.csv")
data_entry = pd.read_csv(data_entry_path)

print("="*70)
print("DATASET INFORMATION")
print("="*70)
print(f"Total images: {len(data_entry):,}")
print(f"Unique patients: {data_entry['Patient ID'].nunique():,}")
print(f"\nDataset columns: {list(data_entry.columns)}")
print(f"\nFirst few rows:")
print(data_entry.head())

# Display label distribution
print("\n" + "="*70)
print("TOP 20 DISEASE COMBINATIONS")
print("="*70)
print(data_entry['Finding Labels'].value_counts().head(20))


DATASET INFORMATION
Total images: 112,120
Unique patients: 30,805

Dataset columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']

First few rows:
        Image Index          Finding Labels  Follow-up #  Patient ID  \
0  00000001_000.png            Cardiomegaly            0           1   
1  00000001_001.png  Cardiomegaly|Emphysema            1           1   
2  00000001_002.png   Cardiomegaly|Effusion            2           1   
3  00000002_000.png              No Finding            0           2   
4  00000003_000.png                  Hernia            0           3   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           58              M            PA                 2682     2749   
1           58              M            PA                 2894     2729   
2           58              M       

In [10]:
# ========================================================================
# SPLIT DATASET WITH SUBSET OPTION (FOR 45GB DATASET)
# ========================================================================

# Check if official split files exist
train_list_path = os.path.join(extract_path, "train_val_list.txt")
test_list_path = os.path.join(extract_path, "test_list.txt")

if os.path.exists(train_list_path) and os.path.exists(test_list_path):
    print("Using official train/test split...")
    train_list = pd.read_csv(train_list_path, header=None)[0].tolist()
    test_list = pd.read_csv(test_list_path, header=None)[0].tolist()
    train_val_data = data_entry[data_entry['Image Index'].isin(train_list)].reset_index(drop=True)
    test_data = data_entry[data_entry['Image Index'].isin(test_list)].reset_index(drop=True)
else:
    print("Official split not found. Creating custom split...")
    train_val_data, test_data = train_test_split(data_entry, test_size=0.3, random_state=42)

# Split train_val into train and validation (80-20)
train_data, val_data = train_test_split(train_val_data, test_size=0.2, random_state=42)

print("\n" + "="*70)
print("ORIGINAL DATASET SPLIT")
print("="*70)
print(f"Training set:   {len(train_data):,} images")
print(f"Validation set: {len(val_data):,} images")
print(f"Test set:       {len(test_data):,} images")

# ========================================================================
# SUBSET OPTION FOR FASTER EXPERIMENTATION (45GB DATASET)
# ========================================================================

USE_SUBSET = True  # Set to False for full training
SUBSET_FRACTION = 0.2  # Use 20% of data for fast experimentation

if USE_SUBSET:
    print("\n" + "⚠️ "*35)
    print(f"USING {SUBSET_FRACTION*100:.0f}% SUBSET FOR FASTER TRAINING")
    print("⚠️ "*35)
    print("\nThis is recommended for:")
    print("  • Testing code and debugging")
    print("  • Hyperparameter tuning")
    print("  • Architecture experiments")
    print(f"\nSet USE_SUBSET=False for full dataset training")
    print("="*70)
    
    train_data = train_data.sample(frac=SUBSET_FRACTION, random_state=42).reset_index(drop=True)
    val_data = val_data.sample(frac=SUBSET_FRACTION, random_state=42).reset_index(drop=True)
    test_data = test_data.sample(frac=SUBSET_FRACTION, random_state=42).reset_index(drop=True)

print("\n" + "="*70)
print("FINAL DATASET SPLIT")
print("="*70)
print(f"Training set:   {len(train_data):,} images")
print(f"Validation set: {len(val_data):,} images")
print(f"Test set:       {len(test_data):,} images")
print("="*70)


Using official train/test split...

ORIGINAL DATASET SPLIT
Training set:   69,219 images
Validation set: 17,305 images
Test set:       25,596 images

⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ 
USING 20% SUBSET FOR FASTER TRAINING
⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ 

This is recommended for:
  • Testing code and debugging
  • Hyperparameter tuning
  • Architecture experiments

Set USE_SUBSET=False for full dataset training

FINAL DATASET SPLIT
Training set:   13,844 images
Validation set: 3,461 images
Test set:       5,119 images


In [11]:
# ========================================================================
# OPTIMIZED AUGMENTATION FOR 45GB DATASET
# ========================================================================

# CRITICAL: Use 128x128 for 3x faster training with 45GB dataset
IMAGE_SIZE = 128  # Change to 224 for higher quality (but slower)

# Simple, fast augmentations
train_transforms = Compose([
    Resize(IMAGE_SIZE, IMAGE_SIZE),
    HorizontalFlip(p=0.5),
    RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.3),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = Compose([
    Resize(IMAGE_SIZE, IMAGE_SIZE),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print("="*70)
print("OPTIMIZED TRANSFORMS FOR LARGE DATASET")
print("="*70)
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Augmentations: Minimal (for speed)")
print("\nTraining transforms:")
print("  ✓ Resize")
print("  ✓ Horizontal flip")
print("  ✓ Brightness/Contrast")
print("  ✓ Normalization")
print("\nSpeed benefit: 3-4x faster than 224x224")
print("="*70)


OPTIMIZED TRANSFORMS FOR LARGE DATASET
Image size: 128x128
Augmentations: Minimal (for speed)

Training transforms:
  ✓ Resize
  ✓ Horizontal flip
  ✓ Brightness/Contrast
  ✓ Normalization

Speed benefit: 3-4x faster than 224x224


In [12]:
# ========================================================================
# CUSTOM DATASET CLASS
# ========================================================================

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, num_classes=14):
        self.dataframe = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.num_classes = num_classes
        
        self.pathologies = [
            'Atelectasis', 'Consolidation', 'Infiltration', 
            'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 
            'Effusion', 'Pneumonia', 'Pleural_Thickening', 
            'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
        ]
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['Image Index']
        
        # Check multiple possible paths
        possible_paths = [
            os.path.join(self.img_dir, 'images', img_name),
            os.path.join(self.img_dir, img_name),
            os.path.join(self.img_dir, 'images_001', 'images', img_name),
            os.path.join(self.img_dir, 'images_002', 'images', img_name),
            os.path.join(self.img_dir, 'images_003', 'images', img_name),
            os.path.join(self.img_dir, 'images_004', 'images', img_name),
            os.path.join(self.img_dir, 'images_005', 'images', img_name),
            os.path.join(self.img_dir, 'images_006', 'images', img_name),
            os.path.join(self.img_dir, 'images_007', 'images', img_name),
            os.path.join(self.img_dir, 'images_008', 'images', img_name),
            os.path.join(self.img_dir, 'images_009', 'images', img_name),
            os.path.join(self.img_dir, 'images_010', 'images', img_name),
            os.path.join(self.img_dir, 'images_011', 'images', img_name),
            os.path.join(self.img_dir, 'images_012', 'images', img_name),
        ]
        
        img_path = None
        for path in possible_paths:
            if os.path.exists(path):
                img_path = path
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found: {img_name}")
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        image = np.array(image)
        
        # Get labels
        labels_str = self.dataframe.iloc[idx]['Finding Labels']
        labels = np.zeros(self.num_classes, dtype=np.float32)
        
        if labels_str != 'No Finding':
            for i, pathology in enumerate(self.pathologies):
                if pathology in labels_str:
                    labels[i] = 1.0
        
        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, torch.tensor(labels)

print("✅ Custom Dataset class defined!")


✅ Custom Dataset class defined!


In [13]:
# ========================================================================
# OPTIMIZED DATALOADERS FOR 45GB DATASET
# ========================================================================

# CRITICAL SETTINGS FOR LARGE DATASET ON SLOW DISK
BATCH_SIZE = 64  # Larger batch size
NUM_WORKERS = 0  # IMPORTANT: 0 is fastest for slow disk/HDD
PIN_MEMORY = False  # Disable when NUM_WORKERS=0

print("="*70)
print("DATALOADER CONFIGURATION FOR 45GB DATASET")
print("="*70)
print(f"Image size:     {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size:     {BATCH_SIZE}")
print(f"Num workers:    {NUM_WORKERS} (optimized for slow disk)")
print(f"Pin memory:     {PIN_MEMORY}")
print("\nOptimizations applied:")
print("  ✓ NUM_WORKERS=0 (single process - faster for HDD)")
print("  ✓ Large batch size (64)")
print("  ✓ Small images (128x128)")
print("="*70)

# Create datasets
train_dataset = ChestXrayDataset(train_data, img_dir=extract_path, transform=train_transforms)
val_dataset = ChestXrayDataset(val_data, img_dir=extract_path, transform=val_transforms)
test_dataset = ChestXrayDataset(test_data, img_dir=extract_path, transform=val_transforms)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("\n" + "="*70)
print("DATALOADERS CREATED")
print("="*70)
print(f"Train batches: {len(train_loader):,} ({len(train_dataset):,} images)")
print(f"Val batches:   {len(val_loader):,} ({len(val_dataset):,} images)")
print(f"Test batches:  {len(test_loader):,} ({len(test_dataset):,} images)")
print("="*70)


DATALOADER CONFIGURATION FOR 45GB DATASET
Image size:     128x128
Batch size:     64
Num workers:    0 (optimized for slow disk)
Pin memory:     False

Optimizations applied:
  ✓ NUM_WORKERS=0 (single process - faster for HDD)
  ✓ Large batch size (64)
  ✓ Small images (128x128)

DATALOADERS CREATED
Train batches: 217 (13,844 images)
Val batches:   55 (3,461 images)
Test batches:  80 (5,119 images)


In [14]:
# ========================================================================
# TEST DATALOADER SPEED
# ========================================================================

def test_dataloader_speed(loader, num_batches=50):
    """Test loading speed"""
    
    print(f"\n🧪 Testing DataLoader speed ({num_batches} batches)...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    start = time.time()
    for i, (images, labels) in enumerate(loader):
        if i >= num_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        if (i + 1) % 10 == 0:
            elapsed = time.time() - start
            speed = (i + 1) / elapsed
            print(f"  Batch {i+1}: {speed:.2f} batch/s")
    
    total_time = time.time() - start
    batches_per_sec = num_batches / total_time
    images_per_sec = batches_per_sec * BATCH_SIZE
    
    print(f"\n📊 DataLoader Performance:")
    print(f"{'='*70}")
    print(f"Batches/sec:       {batches_per_sec:.2f}")
    print(f"Images/sec:        {images_per_sec:.1f}")
    print(f"Seconds/batch:     {total_time/num_batches:.3f}")
    print(f"Epoch time est:    {len(loader)/batches_per_sec/60:.1f} minutes")
    print(f"{'='*70}")
    
    if batches_per_sec < 1.5:
        print(f"\n⚠️ STILL SLOW! Try these:")
        print(f"   1. Copy dataset to faster SSD")
        print(f"   2. Reduce IMAGE_SIZE to 96")
        print(f"   3. Increase BATCH_SIZE to 128")
    elif batches_per_sec < 3.0:
        print(f"\n⚠️ Acceptable but could be faster")
        print(f"   Consider copying to SSD")
    else:
        print(f"\n✅ Good speed!")

# Test speed
test_dataloader_speed(train_loader, num_batches=50)



🧪 Testing DataLoader speed (50 batches)...
  Batch 10: 0.34 batch/s
  Batch 20: 0.32 batch/s
  Batch 30: 0.32 batch/s
  Batch 40: 0.33 batch/s
  Batch 50: 0.33 batch/s

📊 DataLoader Performance:
Batches/sec:       0.32
Images/sec:        20.8
Seconds/batch:     3.083
Epoch time est:    11.1 minutes

⚠️ STILL SLOW! Try these:
   1. Copy dataset to faster SSD
   2. Reduce IMAGE_SIZE to 96
   3. Increase BATCH_SIZE to 128


In [15]:
# ========================================================================
# CALCULATE CLASS WEIGHTS
# ========================================================================

def calculate_class_weights(dataframe, num_classes=14):
    pathologies = [
        'Atelectasis', 'Consolidation', 'Infiltration', 
        'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 
        'Effusion', 'Pneumonia', 'Pleural_Thickening', 
        'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
    ]
    
    total_samples = len(dataframe)
    pos_weights = []
    
    print("\n" + "="*70)
    print("CLASS DISTRIBUTION")
    print("="*70)
    print(f"{'Disease':<25} {'Count':<10} {'%':<10} {'Weight':<10}")
    print("-"*70)
    
    for pathology in pathologies:
        pos_count = dataframe['Finding Labels'].str.contains(pathology, regex=False).sum()
        neg_count = total_samples - pos_count
        weight = neg_count / pos_count if pos_count > 0 else 1.0
        pos_weights.append(weight)
        percentage = (pos_count / total_samples) * 100
        print(f"{pathology:<25} {pos_count:<10} {percentage:<10.2f} {weight:<10.2f}")
    
    print("="*70)
    return torch.tensor(pos_weights, dtype=torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_weights = calculate_class_weights(train_data)
class_weights = class_weights.to(device)

print(f"\n✅ Class weights calculated and moved to {device}")



CLASS DISTRIBUTION
Disease                   Count      %          Weight    
----------------------------------------------------------------------
Atelectasis               1290       9.32       9.73      
Consolidation             439        3.17       30.54     
Infiltration              2244       16.21      5.17      
Pneumothorax              409        2.95       32.85     
Edema                     218        1.57       62.50     
Emphysema                 232        1.68       58.67     
Fibrosis                  204        1.47       66.86     
Effusion                  1374       9.92       9.08      
Pneumonia                 118        0.85       116.32    
Pleural_Thickening        373        2.69       36.12     
Cardiomegaly              265        1.91       51.24     
Nodule                    754        5.45       17.36     
Mass                      661        4.77       19.94     
Hernia                    23         0.17       600.91    

✅ Class weights calcula

In [16]:
# ========================================================================
# MODEL ARCHITECTURE
# ========================================================================

class ChestXrayClassifier(nn.Module):
    def __init__(self, num_classes=14, pretrained=True):
        super(ChestXrayClassifier, self).__init__()
        
        if pretrained:
            self.backbone = EfficientNet.from_pretrained('efficientnet-b0')
        else:
            self.backbone = EfficientNet.from_name('efficientnet-b0')
        
        num_features = self.backbone._fc.in_features
        self.backbone._fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

# Initialize model
model = ChestXrayClassifier(num_classes=14, pretrained=True)
model = model.to(device)

print("\n" + "="*70)
print("MODEL INFORMATION")
print("="*70)
print(f"Model:              EfficientNet-B0")
print(f"Device:             {device}")
print(f"Parameters:         {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params:   {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("="*70)


Loaded pretrained weights for efficientnet-b0

MODEL INFORMATION
Model:              EfficientNet-B0
Device:             cuda
Parameters:         4,025,482
Trainable params:   4,025,482


In [17]:
# ========================================================================
# TRAINING CONFIGURATION
# ========================================================================

# Loss function
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)

# Optimizer
optimizer = AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)

# Scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7)

# Mixed precision
scaler = GradScaler()

# Training parameters
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

# Experiment directory
experiment_name = f"chest_xray_efficientnet_b0_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(f'experiments/{experiment_name}', exist_ok=True)

# TensorBoard
writer = SummaryWriter(f'experiments/{experiment_name}/logs')

# Save config
config = {
    'model': 'EfficientNet-B0',
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'subset': USE_SUBSET,
    'subset_fraction': SUBSET_FRACTION if USE_SUBSET else 1.0,
    'learning_rate': 0.0001,
    'weight_decay': 1e-5,
    'num_epochs': NUM_EPOCHS,
    'optimizer': 'AdamW',
    'loss': 'BCEWithLogitsLoss (Weighted)',
    'scheduler': 'ReduceLROnPlateau',
    'mixed_precision': True,
    'device': str(device)
}

print("\n" + "="*70)
print("TRAINING CONFIGURATION")
print("="*70)
for key, value in config.items():
    print(f"{key:<20}: {value}")
print("="*70)

with open(f'experiments/{experiment_name}/config.json', 'w') as f:
    json.dump(config, f, indent=4)

print(f"\n✅ Configuration saved")



TRAINING CONFIGURATION
model               : EfficientNet-B0
image_size          : 128
batch_size          : 64
num_workers         : 0
subset              : True
subset_fraction     : 0.2
learning_rate       : 0.0001
weight_decay        : 1e-05
num_epochs          : 30
optimizer           : AdamW
loss                : BCEWithLogitsLoss (Weighted)
scheduler           : ReduceLROnPlateau
mixed_precision     : True
device              : cuda

✅ Configuration saved


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\2376986483.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [18]:
# ========================================================================
# OPTIMIZED TRAINING FUNCTIONS
# ========================================================================

def train_epoch(model, loader, criterion, optimizer, device, epoch, scaler):
    model.train()
    running_loss = 0.0
    all_labels = []
    all_predictions = []
    
    print(f"\n🔄 Training Epoch {epoch+1}")
    epoch_start = time.time()
    
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
        if batch_idx % 5 == 0:
            all_labels.append(labels.cpu().numpy())
            predictions = torch.sigmoid(outputs).detach().cpu().numpy()
            all_predictions.append(predictions)
        
        if (batch_idx + 1) % 50 == 0:
            avg_loss = running_loss / (batch_idx + 1)
            elapsed = time.time() - epoch_start
            speed = (batch_idx + 1) / elapsed
            eta = (len(loader) - batch_idx - 1) / speed / 60
            print(f"  [{batch_idx+1}/{len(loader)}] Loss: {avg_loss:.4f} | "
                  f"{speed:.2f} batch/s | ETA: {eta:.1f}min")
    
    if len(all_labels) > 0:
        all_labels = np.vstack(all_labels)
        all_predictions = np.vstack(all_predictions)
        avg_loss = running_loss / len(loader)
        try:
            auc_macro = roc_auc_score(all_labels, all_predictions, average='macro')
            auc_weighted = roc_auc_score(all_labels, all_predictions, average='weighted')
        except:
            auc_macro = 0.0
            auc_weighted = 0.0
    else:
        avg_loss = running_loss / len(loader)
        auc_macro = 0.0
        auc_weighted = 0.0
    
    return avg_loss, auc_macro, auc_weighted


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            all_labels.append(labels.cpu().numpy())
            predictions = torch.sigmoid(outputs).cpu().numpy()
            all_predictions.append(predictions)
    
    all_labels = np.vstack(all_labels)
    all_predictions = np.vstack(all_predictions)
    avg_loss = running_loss / len(loader)
    
    auc_macro = roc_auc_score(all_labels, all_predictions, average='macro')
    auc_weighted = roc_auc_score(all_labels, all_predictions, average='weighted')
    
    per_class_auc = []
    for i in range(all_labels.shape[1]):
        try:
            auc = roc_auc_score(all_labels[:, i], all_predictions[:, i])
            per_class_auc.append(auc)
        except:
            per_class_auc.append(0.0)
    
    return avg_loss, auc_macro, auc_weighted, per_class_auc


def calculate_metrics(labels, predictions, threshold=0.5):
    binary_preds = (predictions >= threshold).astype(int)
    accuracy = accuracy_score(labels.flatten(), binary_preds.flatten())
    f1_macro = f1_score(labels, binary_preds, average='macro', zero_division=0)
    f1_weighted = f1_score(labels, binary_preds, average='weighted', zero_division=0)
    return accuracy, f1_macro, f1_weighted

print("✅ Training functions defined!")


✅ Training functions defined!


In [19]:
# ========================================================================
# MAIN TRAINING LOOP
# ========================================================================

def train_model():
    best_val_auc = 0.0
    early_stop_counter = 0
    training_start_time = time.time()
    
    pathologies = [
        'Atelectasis', 'Consolidation', 'Infiltration', 
        'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 
        'Effusion', 'Pneumonia', 'Pleural_Thickening', 
        'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
    ]
    
    print("\n" + "="*70)
    print("🚀 STARTING TRAINING")
    print("="*70)
    print(f"Configuration: {IMAGE_SIZE}x{IMAGE_SIZE}, Batch {BATCH_SIZE}")
    if USE_SUBSET:
        print(f"Dataset: {SUBSET_FRACTION*100:.0f}% subset")
    print("="*70)
    
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        
        print(f"\n{'='*70}")
        print(f"EPOCH [{epoch+1}/{NUM_EPOCHS}]")
        print(f"{'='*70}")
        
        # Train
        train_loss, train_auc_macro, train_auc_weighted = train_epoch(
            model, train_loader, criterion, optimizer, device, epoch, scaler
        )
        
        # Validate
        val_loss, val_auc_macro, val_auc_weighted, per_class_auc = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update LR
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_auc_macro)
        current_lr = optimizer.param_groups[0]['lr']
        
        if old_lr != current_lr:
            print(f"\n📉 LR: {old_lr:.2e} → {current_lr:.2e}")
        
        epoch_time = time.time() - epoch_start
        total_elapsed = (time.time() - training_start_time) / 3600
        
        # Log
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('AUC/train', train_auc_macro, epoch)
        writer.add_scalar('AUC/val', val_auc_macro, epoch)
        writer.add_scalar('LR', current_lr, epoch)
        
        for pathology, auc in zip(pathologies, per_class_auc):
            writer.add_scalar(f'AUC_class/{pathology}', auc, epoch)
        
        # Print
        print(f"\n📊 Results:")
        print(f"  Train: Loss={train_loss:.4f}, AUC={train_auc_macro:.4f}")
        print(f"  Val:   Loss={val_loss:.4f}, AUC={val_auc_macro:.4f}")
        print(f"  Time: {epoch_time/60:.1f}min | Total: {total_elapsed:.2f}hrs")
        
        if epoch > 0:
            avg_time = (time.time() - training_start_time) / (epoch + 1)
            eta = (avg_time * (NUM_EPOCHS - epoch - 1)) / 3600
            print(f"  ETA: {eta:.2f} hours")
        
        # Per-class (compact)
        print(f"\n  Per-class AUC:")
        for pathology, auc in zip(pathologies, per_class_auc):
            bar = '█' * int(auc * 20)
            print(f"    {pathology:<22}: {auc:.3f} {bar}")
        
        # Save best
        if val_auc_macro > best_val_auc:
            best_val_auc = val_auc_macro
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_auc_macro': val_auc_macro,
                'per_class_auc': per_class_auc,
                'config': config
            }, f'experiments/{experiment_name}/best_model.pth')
            print(f"\n  ✅ Best model saved! AUC: {best_val_auc:.4f}")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            print(f"\n  ⏳ No improvement ({early_stop_counter}/{EARLY_STOPPING_PATIENCE})")
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
        }, f'experiments/{experiment_name}/latest.pth')
        
        if early_stop_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n⚠️ Early stopping at epoch {epoch+1}")
            break
    
    total_time = time.time() - training_start_time
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print(f"Best AUC: {best_val_auc:.4f}")
    print(f"Time: {total_time/3600:.2f} hours")
    print("="*70)
    
    writer.close()
    return best_val_auc

print("✅ Training function ready!")


✅ Training function ready!


In [20]:
# ========================================================================
# START TRAINING
# ========================================================================

print("\n" + "🎬 "*25)
print("STARTING TRAINING")
print("🎬 "*25 + "\n")

best_auc = train_model()

print("\n" + "🎉 "*25)
print(f"Training done! Best AUC: {best_auc:.4f}")
print("🎉 "*25)



🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 
STARTING TRAINING
🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 


🚀 STARTING TRAINING
Configuration: 128x128, Batch 64
Dataset: 20% subset

EPOCH [1/30]

🔄 Training Epoch 1


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [50/217] Loss: 1.3380 | 0.39 batch/s | ETA: 7.2min
  [100/217] Loss: 1.2955 | 0.38 batch/s | ETA: 5.1min
  [150/217] Loss: 1.2975 | 0.38 batch/s | ETA: 2.9min
  [200/217] Loss: 1.2960 | 0.38 batch/s | ETA: 0.7min


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



📊 Results:
  Train: Loss=1.2887, AUC=0.6146
  Val:   Loss=1.2923, AUC=0.6837
  Time: 11.7min | Total: 0.19hrs

  Per-class AUC:
    Atelectasis           : 0.695 █████████████
    Consolidation         : 0.767 ███████████████
    Infiltration          : 0.578 ███████████
    Pneumothorax          : 0.688 █████████████
    Edema                 : 0.864 █████████████████
    Emphysema             : 0.682 █████████████
    Fibrosis              : 0.632 ████████████
    Effusion              : 0.783 ███████████████
    Pneumonia             : 0.659 █████████████
    Pleural_Thickening    : 0.699 █████████████
    Cardiomegaly          : 0.714 ██████████████
    Nodule                : 0.576 ███████████
    Mass                  : 0.616 ████████████
    Hernia                : 0.619 ████████████

  ✅ Best model saved! AUC: 0.6837

EPOCH [2/30]

🔄 Training Epoch 2


...


📊 Results:
  Train: Loss=0.4981, AUC=0.9359
  Val:   Loss=2.1567, AUC=0.7307
  Time: 6.3min | Total: 1.96hrs
  ETA: 2.24 hours

  Per-class AUC:
    Atelectasis           : 0.773 ███████████████
    Consolidation         : 0.764 ███████████████
    Infiltration          : 0.649 ████████████
    Pneumothorax          : 0.738 ██████████████
    Edema                 : 0.889 █████████████████
    Emphysema             : 0.751 ███████████████
    Fibrosis              : 0.641 ████████████
    Effusion              : 0.858 █████████████████
    Pneumonia             : 0.613 ████████████
    Pleural_Thickening    : 0.736 ██████████████
    Cardiomegaly          : 0.800 ███████████████
    Nodule                : 0.682 █████████████
    Mass                  : 0.738 ██████████████
    Hernia                : 0.597 ███████████

  ⏳ No improvement (6/7)

EPOCH [15/30]

🔄 Training Epoch 15


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [50/217] Loss: 0.4760 | 0.73 batch/s | ETA: 3.8min
  [100/217] Loss: 0.4683 | 0.72 batch/s | ETA: 2.7min
  [150/217] Loss: 0.4719 | 0.73 batch/s | ETA: 1.5min
  [200/217] Loss: 0.4731 | 0.74 batch/s | ETA: 0.4min


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



📊 Results:
  Train: Loss=0.4752, AUC=0.9384
  Val:   Loss=2.1471, AUC=0.7328
  Time: 6.0min | Total: 2.06hrs
  ETA: 2.06 hours

  Per-class AUC:
    Atelectasis           : 0.771 ███████████████
    Consolidation         : 0.762 ███████████████
    Infiltration          : 0.649 ████████████
    Pneumothorax          : 0.731 ██████████████
    Edema                 : 0.893 █████████████████
    Emphysema             : 0.740 ██████████████
    Fibrosis              : 0.644 ████████████
    Effusion              : 0.861 █████████████████
    Pneumonia             : 0.620 ████████████
    Pleural_Thickening    : 0.749 ██████████████
    Cardiomegaly          : 0.814 ████████████████
    Nodule                : 0.687 █████████████
    Mass                  : 0.737 ██████████████
    Hernia                : 0.600 ███████████

  ⏳ No improvement (7/7)

⚠️ Early stopping at epoch 15

✅ TRAINING COMPLETE!
Best AUC: 0.7612
Time: 2.06 hours

🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 
Tra

In [24]:
# ========================================================================
# FIX: EVALUATE TEST SET (Corrected autocast syntax)
# ========================================================================

def evaluate_test_set(model_path, test_loader, device):
    """Comprehensive evaluation on test set"""
    
    pathologies = [
        'Atelectasis', 'Consolidation', 'Infiltration', 
        'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 
        'Effusion', 'Pneumonia', 'Pleural_Thickening', 
        'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
    ]
    
    print("\n" + "="*70)
    print("🧪 EVALUATING TEST SET")
    print("="*70)
    
    # Load checkpoint with weights_only=False for PyTorch 2.6+
    checkpoint = torch.load(model_path, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"Loaded model from epoch {checkpoint['epoch']}")
    print(f"Val AUC: {checkpoint['val_auc_macro']:.4f}")
    
    all_labels = []
    all_predictions = []
    
    print("\nProcessing test set...")
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            # FIX: Use the original autocast() without arguments
            with autocast():
                outputs = model(images)
                predictions = torch.sigmoid(outputs).cpu().numpy()
            
            all_labels.append(labels.cpu().numpy())
            all_predictions.append(predictions)
            
            if (batch_idx + 1) % 20 == 0:
                print(f"  {batch_idx+1}/{len(test_loader)} batches processed")
    
    # Combine predictions
    all_labels = np.vstack(all_labels)
    all_predictions = np.vstack(all_predictions)
    
    # Calculate metrics
    auc_macro = roc_auc_score(all_labels, all_predictions, average='macro')
    auc_weighted = roc_auc_score(all_labels, all_predictions, average='weighted')
    
    # Per-class AUC
    per_class_auc = []
    for i in range(all_labels.shape[1]):
        try:
            auc = roc_auc_score(all_labels[:, i], all_predictions[:, i])
            per_class_auc.append(auc)
        except:
            per_class_auc.append(0.0)
    
    # Additional metrics
    accuracy, f1_macro, f1_weighted = calculate_metrics(all_labels, all_predictions)
    
    # Print results
    print("\n" + "="*70)
    print("📊 TEST SET RESULTS")
    print("="*70)
    print(f"AUC (macro):     {auc_macro:.4f}")
    print(f"AUC (weighted):  {auc_weighted:.4f}")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"F1 (macro):      {f1_macro:.4f}")
    print(f"F1 (weighted):   {f1_weighted:.4f}")
    
    print("\n" + "="*70)
    print("📈 PER-CLASS AUC ON TEST SET")
    print("="*70)
    print(f"{'Disease':<25} {'AUC':<10} {'Performance'}")
    print("-"*70)
    
    for pathology, auc in zip(pathologies, per_class_auc):
        bar = '█' * int(auc * 40)
        status = "🟢" if auc >= 0.80 else "🟡" if auc >= 0.70 else "🔴"
        print(f"{pathology:<25} {auc:.4f}     {status} {bar}")
    
    print("="*70)
    
    # Save results
    results = {
        'test_auc_macro': float(auc_macro),
        'test_auc_weighted': float(auc_weighted),
        'test_accuracy': float(accuracy),
        'test_f1_macro': float(f1_macro),
        'test_f1_weighted': float(f1_weighted),
        'per_class_auc': {p: float(a) for p, a in zip(pathologies, per_class_auc)}
    }
    
    with open(f'experiments/{experiment_name}/test_results.json', 'w') as f:
        json.dump(results, f, indent=4)
    
    print(f"\n✅ Results saved to: experiments/{experiment_name}/test_results.json")
    
    return results

# Run evaluation
test_results = evaluate_test_set(
    f'experiments/{experiment_name}/best_model.pth',
    test_loader,
    device
)



🧪 EVALUATING TEST SET
Loaded model from epoch 8
Val AUC: 0.7612

Processing test set...


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\3974932263.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  20/80 batches processed
  40/80 batches processed
  60/80 batches processed
  80/80 batches processed

📊 TEST SET RESULTS
AUC (macro):     0.7163
AUC (weighted):  0.7165
Accuracy:        0.6769
F1 (macro):      0.2028
F1 (weighted):   0.2981

📈 PER-CLASS AUC ON TEST SET
Disease                   AUC        Performance
----------------------------------------------------------------------
Atelectasis               0.7080     🟡 ████████████████████████████
Consolidation             0.6926     🔴 ███████████████████████████
Infiltration              0.6735     🔴 ██████████████████████████
Pneumothorax              0.7709     🟡 ██████████████████████████████
Edema                     0.7561     🟡 ██████████████████████████████
Emphysema                 0.7376     🟡 █████████████████████████████
Fibrosis                  0.7403     🟡 █████████████████████████████
Effusion                  0.7603     🟡 ██████████████████████████████
Pneumonia                 0.5716     🔴 ███████████████████

In [26]:
# ========================================================================
# STEP 1: CONFIGURE FOR FULL DATASET TRAINING
# ========================================================================

print("\n" + "🚀 "*35)
print("PHASE 1: TRAINING ON FULL DATASET")
print("🚀 "*35 + "\n")

print("="*70)
print("CONFIGURATION")
print("="*70)

# CRITICAL SETTINGS FOR FULL DATASET
USE_SUBSET = False  # Changed from True
IMAGE_SIZE = 128  # Keep 128 for reasonable speed
BATCH_SIZE = 64
NUM_WORKERS = 0
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

print(f"Dataset:      100% (FULL)")
print(f"Image size:   {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size:   {BATCH_SIZE}")
print(f"Epochs:       {NUM_EPOCHS}")
print(f"Expected:     6-8 hours total")
print("="*70)

# Reload original data
print("\n📂 Loading full dataset...")

# Reload data entry
data_entry = pd.read_csv(os.path.join(extract_path, "Data_Entry_2017.csv"))

# Recreate splits
train_list_path = os.path.join(extract_path, "train_val_list.txt")
test_list_path = os.path.join(extract_path, "test_list.txt")

if os.path.exists(train_list_path) and os.path.exists(test_list_path):
    train_list = pd.read_csv(train_list_path, header=None)[0].tolist()
    test_list = pd.read_csv(test_list_path, header=None)[0].tolist()
    train_val_data = data_entry[data_entry['Image Index'].isin(train_list)].reset_index(drop=True)
    test_data = data_entry[data_entry['Image Index'].isin(test_list)].reset_index(drop=True)
else:
    train_val_data, test_data = train_test_split(data_entry, test_size=0.3, random_state=42)

# Split into train and validation
train_data, val_data = train_test_split(train_val_data, test_size=0.2, random_state=42)

print(f"\n✅ Full dataset loaded:")
print(f"   Train:      {len(train_data):,} images")
print(f"   Validation: {len(val_data):,} images")
print(f"   Test:       {len(test_data):,} images")
print(f"   Total:      {len(data_entry):,} images")



🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 
PHASE 1: TRAINING ON FULL DATASET
🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 🚀 

CONFIGURATION
Dataset:      100% (FULL)
Image size:   128x128
Batch size:   64
Epochs:       30
Expected:     6-8 hours total

📂 Loading full dataset...

✅ Full dataset loaded:
   Train:      69,219 images
   Validation: 17,305 images
   Test:       25,596 images
   Total:      112,120 images


In [27]:
# ========================================================================
# STEP 2: CREATE DATALOADERS FOR FULL DATASET
# ========================================================================

print("\n📦 Creating dataloaders for full dataset...")

# Transforms (same as before)
train_transforms = Compose([
    Resize(IMAGE_SIZE, IMAGE_SIZE),
    HorizontalFlip(p=0.5),
    RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.3),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = Compose([
    Resize(IMAGE_SIZE, IMAGE_SIZE),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Create datasets
train_dataset = ChestXrayDataset(train_data, img_dir=extract_path, transform=train_transforms)
val_dataset = ChestXrayDataset(val_data, img_dir=extract_path, transform=val_transforms)
test_dataset = ChestXrayDataset(test_data, img_dir=extract_path, transform=val_transforms)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"\n✅ Dataloaders created:")
print(f"   Train batches: {len(train_loader):,}")
print(f"   Val batches:   {len(val_loader):,}")
print(f"   Test batches:  {len(test_loader):,}")

# Calculate class weights
print("\n⚖️ Calculating class weights...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_weights = calculate_class_weights(train_data)
class_weights = class_weights.to(device)



📦 Creating dataloaders for full dataset...

✅ Dataloaders created:
   Train batches: 1,082
   Val batches:   271
   Test batches:  400

⚖️ Calculating class weights...

CLASS DISTRIBUTION
Disease                   Count      %          Weight    
----------------------------------------------------------------------
Atelectasis               6575       9.50       9.53      
Consolidation             2244       3.24       29.85     
Infiltration              11034      15.94      5.27      
Pneumothorax              2098       3.03       31.99     
Edema                     1122       1.62       60.69     
Emphysema                 1127       1.63       60.42     
Fibrosis                  984        1.42       69.34     
Effusion                  6896       9.96       9.04      
Pneumonia                 696        1.01       98.45     
Pleural_Thickening        1765       2.55       38.22     
Cardiomegaly              1348       1.95       50.35     
Nodule                    3787  

In [28]:
# ========================================================================
# STEP 3: INITIALIZE NEW MODEL FOR FULL TRAINING
# ========================================================================

print("\n🏗️ Initializing fresh model...")

# Create new model
model = ChestXrayClassifier(num_classes=14, pretrained=True)
model = model.to(device)

# Setup training components
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
optimizer = AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7)
scaler = GradScaler()

# Create new experiment directory
experiment_name_full = f"chest_xray_FULL_dataset_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(f'experiments/{experiment_name_full}', exist_ok=True)
writer = SummaryWriter(f'experiments/{experiment_name_full}/logs')

# Save config
config_full = {
    'model': 'EfficientNet-B0',
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'subset': False,
    'dataset_size': 'Full (100%)',
    'learning_rate': 0.0001,
    'num_epochs': NUM_EPOCHS,
    'device': str(device)
}

with open(f'experiments/{experiment_name_full}/config.json', 'w') as f:
    json.dump(config_full, f, indent=4)

print(f"\n✅ Model initialized!")
print(f"✅ Experiment: {experiment_name_full}")



🏗️ Initializing fresh model...
Loaded pretrained weights for efficientnet-b0

✅ Model initialized!
✅ Experiment: chest_xray_FULL_dataset_20251021_230048


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\1078069800.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# ========================================================================
# STEP 4: START FULL DATASET TRAINING
# ========================================================================

print("\n" + "="*70)
print("🚀 STARTING FULL DATASET TRAINING")
print("="*70)
print("\nThis will take 6-8 hours. You can:")
print("  • Let it run overnight")
print("  • Monitor progress in real-time")
print("  • Check TensorBoard logs")
print("="*70)

# Train model (using same train_model function)
def train_model_full():
    """Train on full dataset"""
    
    best_val_auc = 0.0
    early_stop_counter = 0
    training_start_time = time.time()
    
    pathologies = [
        'Atelectasis', 'Consolidation', 'Infiltration', 
        'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 
        'Effusion', 'Pneumonia', 'Pleural_Thickening', 
        'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
    ]
    
    print(f"\n🎯 Target: 78-82% AUC (compared to your 71.6% with 20% data)")
    print(f"📊 Training on {len(train_data):,} images")
    print("="*70)
    
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        
        print(f"\n{'='*70}")
        print(f"EPOCH [{epoch+1}/{NUM_EPOCHS}]")
        print(f"{'='*70}")
        
        # Train
        train_loss, train_auc_macro, train_auc_weighted = train_epoch(
            model, train_loader, criterion, optimizer, device, epoch, scaler
        )
        
        # Validate
        val_loss, val_auc_macro, val_auc_weighted, per_class_auc = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update LR
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_auc_macro)
        current_lr = optimizer.param_groups[0]['lr']
        
        if old_lr != current_lr:
            print(f"\n📉 LR: {old_lr:.2e} → {current_lr:.2e}")
        
        epoch_time = time.time() - epoch_start
        total_elapsed = (time.time() - training_start_time) / 3600
        
        # Log
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('AUC/train', train_auc_macro, epoch)
        writer.add_scalar('AUC/val', val_auc_macro, epoch)
        writer.add_scalar('LR', current_lr, epoch)
        
        # Print
        print(f"\n📊 Results:")
        print(f"  Train: Loss={train_loss:.4f}, AUC={train_auc_macro:.4f}")
        print(f"  Val:   Loss={val_loss:.4f}, AUC={val_auc_macro:.4f}")
        print(f"  Time: {epoch_time/60:.1f}min | Total: {total_elapsed:.2f}hrs")
        
        if epoch > 0:
            avg_time = (time.time() - training_start_time) / (epoch + 1)
            eta = (avg_time * (NUM_EPOCHS - epoch - 1)) / 3600
            print(f"  ETA: {eta:.2f} hours")
        
        # Compact per-class display
        print(f"\n  Top 5 diseases:")
        sorted_aucs = sorted(zip(pathologies, per_class_auc), key=lambda x: x[1], reverse=True)
        for path, auc in sorted_aucs[:5]:
            print(f"    {path:<20}: {auc:.3f}")
        
        # Save best
        if val_auc_macro > best_val_auc:
            best_val_auc = val_auc_macro
            improvement = val_auc_macro - 0.7163  # Compare to subset model
            
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_auc_macro': val_auc_macro,
                'per_class_auc': per_class_auc,
                'config': config_full
            }, f'experiments/{experiment_name_full}/best_model.pth')
            
            print(f"\n  ✅ Best model saved! AUC: {best_val_auc:.4f}")
            print(f"     Improvement vs subset: +{improvement*100:.2f}%")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            print(f"\n  ⏳ No improvement ({early_stop_counter}/{EARLY_STOPPING_PATIENCE})")
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
        }, f'experiments/{experiment_name_full}/latest.pth')
        
        if early_stop_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n⚠️ Early stopping at epoch {epoch+1}")
            break
    
    total_time = time.time() - training_start_time
    
    print("\n" + "="*70)
    print("✅ FULL DATASET TRAINING COMPLETE!")
    print(f"Best AUC: {best_val_auc:.4f}")
    print(f"Time: {total_time/3600:.2f} hours")
    print(f"Improvement: +{(best_val_auc - 0.7163)*100:.2f}% vs subset")
    print("="*70)
    
    writer.close()
    return best_val_auc

# START TRAINING
print("\n⏰ Starting now: " + time.strftime('%I:%M %p'))
print("🎯 Expected completion: " + time.strftime('%I:%M %p', time.localtime(time.time() + 28800)))
print("\n" + "🎬 "*35)

best_auc_full = train_model_full()

print("\n" + "🎉 "*35)



🚀 STARTING FULL DATASET TRAINING

This will take 6-8 hours. You can:
  • Let it run overnight
  • Monitor progress in real-time
  • Check TensorBoard logs

⏰ Starting now: 11:01 PM
🎯 Expected completion: 07:01 AM

🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 🎬 

🎯 Target: 78-82% AUC (compared to your 71.6% with 20% data)
📊 Training on 69,219 images

EPOCH [1/30]

🔄 Training Epoch 1


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [50/1082] Loss: 1.3169 | 0.24 batch/s | ETA: 71.4min
  [100/1082] Loss: 1.3029 | 0.22 batch/s | ETA: 73.6min
  [150/1082] Loss: 1.2914 | 0.22 batch/s | ETA: 69.4min
  [200/1082] Loss: 1.2912 | 0.22 batch/s | ETA: 65.5min
  [250/1082] Loss: 1.2958 | 0.22 batch/s | ETA: 63.1min
  [300/1082] Loss: 1.2773 | 0.22 batch/s | ETA: 59.2min
  [350/1082] Loss: 1.2948 | 0.23 batch/s | ETA: 53.3min
  [400/1082] Loss: 1.2971 | 0.24 batch/s | ETA: 48.0min
  [450/1082] Loss: 1.2861 | 0.24 batch/s | ETA: 43.1min
  [500/1082] Loss: 1.2696 | 0.25 batch/s | ETA: 39.1min
  [550/1082] Loss: 1.2636 | 0.25 batch/s | ETA: 35.3min
  [600/1082] Loss: 1.2549 | 0.25 batch/s | ETA: 31.7min
  [650/1082] Loss: 1.2474 | 0.25 batch/s | ETA: 28.3min
  [700/1082] Loss: 1.2392 | 0.25 batch/s | ETA: 25.0min
  [750/1082] Loss: 1.2385 | 0.26 batch/s | ETA: 21.7min
  [800/1082] Loss: 1.2358 | 0.26 batch/s | ETA: 18.4min
  [850/1082] Loss: 1.2341 | 0.26 batch/s | ETA: 15.0min
  [900/1082] Loss: 1.2273 | 0.26 batch/s | ETA: 1

C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



📊 Results:
  Train: Loss=1.2136, AUC=0.7092
  Val:   Loss=1.1452, AUC=0.7704
  Time: 86.6min | Total: 1.44hrs

  Top 5 diseases:
    Edema               : 0.883
    Effusion            : 0.852
    Hernia              : 0.852
    Cardiomegaly        : 0.830
    Emphysema           : 0.806

  ✅ Best model saved! AUC: 0.7704
     Improvement vs subset: +5.41%

EPOCH [2/30]

🔄 Training Epoch 2


...

  [50/1082] Loss: 0.8863 | 0.29 batch/s | ETA: 58.7min
  [100/1082] Loss: 0.8830 | 0.29 batch/s | ETA: 56.7min
  [150/1082] Loss: 0.8956 | 0.28 batch/s | ETA: 54.7min
  [200/1082] Loss: 0.8887 | 0.28 batch/s | ETA: 52.0min
  [250/1082] Loss: 0.8964 | 0.28 batch/s | ETA: 48.9min
  [300/1082] Loss: 0.8946 | 0.28 batch/s | ETA: 46.0min
  [350/1082] Loss: 0.9009 | 0.28 batch/s | ETA: 43.1min
  [400/1082] Loss: 0.9134 | 0.28 batch/s | ETA: 40.2min
  [450/1082] Loss: 0.9098 | 0.28 batch/s | ETA: 37.2min
  [500/1082] Loss: 0.9134 | 0.28 batch/s | ETA: 34.3min
  [550/1082] Loss: 0.9160 | 0.28 batch/s | ETA: 31.3min
  [600/1082] Loss: 0.9155 | 0.28 batch/s | ETA: 28.4min
  [650/1082] Loss: 0.9199 | 0.28 batch/s | ETA: 25.5min
  [700/1082] Loss: 0.9185 | 0.28 batch/s | ETA: 22.6min
  [750/1082] Loss: 0.9210 | 0.28 batch/s | ETA: 19.7min
  [800/1082] Loss: 0.9211 | 0.28 batch/s | ETA: 16.7min
  [850/1082] Loss: 0.9168 | 0.28 batch/s | ETA: 13.7min
  [900/1082] Loss: 0.9174 | 0.28 batch/s | ETA: 1

C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



📊 Results:
  Train: Loss=0.9185, AUC=0.8391
  Val:   Loss=1.2081, AUC=0.7973
  Time: 80.8min | Total: 5.63hrs
  ETA: 36.62 hours

  Top 5 diseases:
    Edema               : 0.896
    Effusion            : 0.877
    Emphysema           : 0.872
    Cardiomegaly        : 0.869
    Pneumothorax        : 0.838

  ⏳ No improvement (1/7)

EPOCH [5/30]

🔄 Training Epoch 5


C:\Users\91850\AppData\Local\Temp\ipykernel_7872\559233409.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [50/1082] Loss: 0.8705 | 0.27 batch/s | ETA: 64.2min


In [ ]:
# ========================================================================
# OPTIONAL: PROGRESS MONITOR (Run this in a separate cell while training)
# ========================================================================

import time as time_module

def monitor_training(experiment_name, check_interval=300):
    """Monitor training progress every 5 minutes"""
    
    print(f"📊 Monitoring: {experiment_name}")
    print(f"Checking every {check_interval//60} minutes...")
    print("Press Ctrl+C to stop monitoring\n")
    
    try:
        while True:
            # Check if best model exists
            model_path = f'experiments/{experiment_name}/best_model.pth'
            if os.path.exists(model_path):
                checkpoint = torch.load(model_path, weights_only=False)
                print(f"[{time_module.strftime('%I:%M %p')}] "
                      f"Epoch {checkpoint['epoch']}: "
                      f"Val AUC = {checkpoint['val_auc_macro']:.4f}")
            else:
                print(f"[{time_module.strftime('%I:%M %p')}] Training not started yet...")
            
            time_module.sleep(check_interval)
    except KeyboardInterrupt:
        print("\n✅ Monitoring stopped")

# Uncomment to use:
monitor_training(experiment_name_full)
